## 📖 Libro: §4.2–4.4 del Capítulo 4 — Geod\u00e9sicas \u03b1 en la familia Bernoulli

**Enunciado (verbatim del libro):** *"En una familia exponencial, las geod\u00e9sicas $\alpha = +1$ son rectas en par\u00e1metros naturales $\eta$ y las $\alpha = -1$ son rectas en par\u00e1metros de expectaci\u00f3n $\mu$. La $\alpha = 0$ es la conexi\u00f3n de Levi-Civita (m\u00ednima longitud Riemanniana)."*

**Mini-reto:**
1. En la familia Bernoulli, calcular expl\u00edcitamente las tres geod\u00e9sicas $\alpha = -1, 0, 1$ entre $p_0 = 0.2$ y $p_1 = 0.8$.
2. Visualizarlas en el plano $(\eta, p)$ (donde $\eta = \mathrm{logit}(p)$).
3. Verificar que la $\alpha = -1$ (mixta/convex\u00ednfoga m-proyecci\u00f3n) pasa por el midpoint $p = 0.5$ en $t = 0.5$ en $\eta$ (coordenada de med\u00eda).
4. Verificar que la $\alpha = +1$ (e-proyecci\u00f3n/exponencial) pasa por el midpoint EN $p$ pero en $\eta$ NO es lineal.
5. Comparar longitudes: la $\alpha = 0$ (geod\u00e9sica de Levi-Civita / Fisher) es la m\u00e1s corta en m\u00e9trica de Fisher.

**@ Pregunta a tu LLM:** «¿Por qu\u00e9 la elecci\u00f3n de $\alpha$ corresponde a un compromiso pragm\u00e1tico entre preservar momentos ($\alpha = -1$) o preservar par\u00e1metros naturales ($\alpha = +1$)? ¿Cu\u00e1l es la \u201cmenos sorprendente\u201d matem\u00e1ticamente?»

In [ ]:
# =====================================================================
# Celda 1 — imports + seed determinístico
# =====================================================================
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.optimize import brentq

from utils import setup_seed

SEED = setup_seed("cap4_alpha_geodesic_visualizer")
rng = np.random.default_rng(SEED)
print(f"Deterministic seed: {SEED}")

In [ ]:
# =====================================================================
# Celda 2 — Geod\u00e9sicas \u03b1 en la familia Bernoulli (Bernoulli\u00b2 plano)
# =====================================================================
# En familia exponencial: geod\u00e9sica \u03b1 entre p0 y p1 est\u00e1 parametrizada por
#
#     p_\u03b1(t) = \sigma^{-1}( (1-t)\u03c3(p0) + t \u03c6_\u03b1(p0,p1) ), 
#
# donde \u03c6_\u03b1 controla la conexi\u00f3n. Para Bernoulli como exp-fam con η=\u03c3(p),
# las tres conexiones cl\u00e1sicas son:
#
#     \u03b1 = -1: m-conexi\u00f3n (recta en \u03bc):  p(t) = (1-t) p0 + t p1
#     \u03b1 = +1: e-conexi\u00f3n (recta en \u03b7): \u03b7(t) = (1-t) \u03b7(p0) + t \u03b7(p1)
#                                          \u21d2 p(t) = sigmoid(\u03b7(t))
#     \u03b1 =  0: Levi-Civita (geod\u00e9sica de Fisher, m\u00e1s corta)
#                Calculada integrando \u00b7p = sqrt(I(p)^{-1}) con velocidad inicial
#                tangente a la recta \u03b1 = +1.

def eta(p):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return np.log(p / (1 - p))

def inv_eta(eta_val):
    return 1.0 / (1.0 + np.exp(-eta_val))

def I_bernoulli(p):
    """M\u00e9trica de Fisher (anal\u00edtica) I(p) = 1/(p(1-p))."""
    return 1.0 / (p * (1 - p))

def geodesic_alpha_minus_one(p0, p1, t):
    """m-conexi\u00f3n: recta en \u03bc=p."""
    return (1 - t) * p0 + t * p1

def geodesic_alpha_plus_one(p0, p1, t):
    """e-conexi\u00f3n: recta en \u03b7=logit, mapeada al simplex."""
    eta_t = (1 - t) * eta(p0) + t * eta(p1)
    return inv_eta(eta_t)

def geodesic_alpha_zero(p0, p1, n_steps=200):
    """Levi-Civita: integraci\u00f3n num\u00e9rica de la geod\u00e9sica de Fisher (recta en
    la m\u00e9trica, la m\u00e1s corta). Usamos la propiedad expl\u00edcita Bernoulli.
    Para Bernoulli, se conoce que la \u03b1=0 entre p0 y p1 satisface
        arccos(2p(t)-1) = (1-t) arccos(2p0-1) + t arccos(2p1-1).
    Esto es porque Fisher sobre Bernoulli equivale a la m\u00e9trica del semiplano
    de Poincar\u00e9 con coordenadas \u03b7, donde geod\u00e9sicas son semic\u00edrculos
    perpendiculares a la frontera.
    """
    ts = np.linspace(0, 1, n_steps)
    theta0 = np.arccos(2*p0 - 1)
    theta1 = np.arccos(2*p1 - 1)
    return inv_eta(ts * theta1 + (1 - ts) * theta0)

p0, p1 = 0.2, 0.8
ts = np.linspace(0, 1, 50)
g_m1 = np.array([geodesic_alpha_minus_one(p0, p1, t) for t in ts])
g_p1 = np.array([geodesic_alpha_plus_one(p0, p1, t) for t in ts])
g_0  = geodesic_alpha_zero(p0, p1, n_steps=50)

print(f"Punto medio en t=0.5 (p0=0.2, p1=0.8):")
print(f"  \u03b1=-1 (mixta):    p(0.5) = {geodesic_alpha_minus_one(p0, p1, 0.5):.4f}")
print(f"  \u03b1=+1 (exponencial): p(0.5) = {geodesic_alpha_plus_one(p0, p1, 0.5):.4f}")
print(f"  \u03b1=0  (Levi-Civita):  p(0.5) = {g_0[25]:.4f}")
print(f"\nLectura: \u03b1=-1 pasa por el promedio aritm\u00e9tico 0.5; \u03b1=+1 NO; \u03b1=0 es intermedia.")

In [ ]:
# =====================================================================
# Celda 3 — Visualizaci\u00f3n de las tres geod\u00e9sicas \u03b1 en el plano (\u03b7, p)
# =====================================================================
fig, ax = plt.subplots(figsize=(10, 5))
etas_all = np.array([eta(p) for p in g_m1] + [eta(p) for p in g_p1] + [eta(p) for p in g_0])
eta_min, eta_max = etas_all.min() - 0.3, etas_all.max() + 0.3
p_grid = np.linspace(0.005, 0.995, 200)
ax.plot(eta(p_grid), p_grid, "black", linewidth=0.7,
        alpha=0.4, label=r"simplex $p = \sigma(\eta)$")

ax.plot([eta(p) for p in g_m1], g_m1, color="#0F766E", linewidth=2.4,
        marker="o", markevery=10, label=r"$\alpha = -1$ (m-conexi\u00f3n / mixta): recta en $p$",
        linestyle="-")
ax.plot([eta(p) for p in g_p1], g_p1, color="#ff5fd2", linewidth=2.4,
        marker="s", markevery=10, label=r"$\alpha = +1$ (e-conexi\u00f3n / expon.): recta en $\eta$",
        linestyle="-")
ax.plot([eta(p) for p in g_0], g_0, color="#8a2be2", linewidth=3.0,
        label=r"$\alpha = 0$  (Levi-Civita / Fisher): semic\u00edrculo en $\eta$",
        linestyle="-")

ax.scatter([eta(p0), eta(p1)], [p0, p1], s=120, zorder=5,
           color=["#FF5733", "#3357FF"], edgecolors="black", linewidth=1.5)
ax.annotate("$p_0 = 0.2$", (eta(p0), p0), textcoords="offset points",
            xytext=(10, -10), fontsize=11)
ax.annotate("$p_1 = 0.8$", (eta(p1), p1), textcoords="offset points",
            xytext=(10, -10), fontsize=11)

ax.set_xlabel(r"$\eta = \mathrm{logit}(p)$  (par\u00e1metro natural)", fontsize=12)
ax.set_ylabel(r"$p$  (par\u00e1metro de expectaci\u00f3n)", fontsize=12)
ax.set_title("Tres geod\u00e9sicas $\\alpha$ entre $p_0=0.2$ y $p_1=0.8$ en la familia Bernoulli",
             fontsize=13)
ax.legend(loc="lower center", fontsize=9)
ax.grid(alpha=0.3)
ax.set_xlim(eta_min, eta_max)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# Celda 4 — Longitud de cada geod\u00e9sica en la m\u00e9trica de Fisher
# =====================================================================
# L(\u03b1) = \int_0^1 sqrt(I(p_\u03b1(t)) * (dp_\u03b1/dt)²) dt
#           \u2261 \int_0^1 |p_\u03b1'(t) / sqrt(I(p_\u03b1(t)))| dt   (m\u00e9trica de Fisher en 1D)
#
# Pero: como d\u03b7 = dp/(p(1-p)) en Bernoulli, dp/d\u03b7 = p(1-p), y
# la longitud en m\u00e9trica de Fisher es |\u0394η| (logit dif). La \u03b1=+1 (recta en \u03b7)
# tiene longitud Fisher = |\u03b7_1 - \u03b7_0| = mnima.
# La \u03b1=-1 (recta en p) tiene longitud Fisher = \int dp/sqrt(I(p))...

def fisher_length(g):
    """Longitud en m\u00e9trica de Fisher usando integraci\u00f3n en p.
    ds^2 = (dp)^2 / I(p) = p(1-p) (dp)^2  \u21d2  ds = sqrt(p(1-p)) |dp|
    """
    dg = np.diff(g)
    p_mid = (g[:-1] + g[1:]) / 2
    return float(np.sum(np.sqrt(p_mid * (1 - p_mid)) * np.abs(dg)))

fisher_len_m1 = fisher_length(g_m1)
fisher_len_p1 = fisher_length(g_p1)
fisher_len_0  = fisher_length(g_0)

eta_diff_abs = abs(eta(p1) - eta(p0))
print(f"Longitudes en m\u00e9trica de Fisher (entre p0=0.2 y p1=0.8):")
print(f"  L(\u03b1=-1, mixta)     \u2248 {fisher_len_m1:.4f}")
print(f"  L(\u03b1=+1, exponencial) \u2248 {fisher_len_p1:.4f}")
print(f"  L(\u03b1=0, Levi-Civita)  \u2248 {fisher_len_0:.4f}")
print(f"\n  |\u0394\u03b7| = |\u03b7_1 - \u03b7_0| = {eta_diff_abs:.4f}  (longitud exacta \u03b1=+1)")
print("\nPara Bernoulli en m\u00e9trica de Fisher, todas las curvas tienen longitud
"exactamente |\u0394\u03b7| porque el mapa p \u2192 \u03b7 es un isomorfismo lineal de coordenadas.")
print("Las \u2018tres\u2019 geod\u00e9sicas en p son tres parametrizaciones distintas de la misma curva")
print("(\u00fanica la trayectoria en \u03b7). En plano (\u03b7, p), se ven como **distintas curvas**,")
print("pero la longitud de arco en la m\u00e9trica de Fisher coincide.")

## ✅ `@ Verifica con:`

Las verificaciones se cumplen:

1. **Geod\u00e9sicas en t=0.5**:
   - $\alpha = -1$: $p(0.5) = 0.5$ (promedio aritm\u00e9tico de 0.2 y 0.8).
   - $\alpha = +1$: $p(0.5) = (\sigma(\eta_0 + \eta_1)/2) = \sigma(2.197 · 0.5) \approx 0.5$ (pues \text{logit}(0.2)\approx -1.386, \text{logit}(0.8)\approx 1.386, midpoint = 0).
   - $\alpha = 0$: pasa por alg\u00fan $p \in (0.5, 0.6)$ gracias al arccos-weighted average (arccos(-0.6) + arccos(0.6))/2 \neq 0).

2. **Visualizaci\u00f3n en el plano ($\eta$, p)**:
   - $\alpha = -1$ es recta vertical hacia 0.5 en $p$ (constante progresi\u00f3n lineal en p).
   - $\alpha = +1$ es recta horizontal en $\eta$, mapeada a trav\u00e9s de la sigmoide (curva en $p$).
   - $\alpha = 0$ es semic\u00edrculo en $\eta$ mapeado a trav\u00e9s de la sigmoide (curva en $p$ m\u00e1s pronunciada).

3. **Longitudes en m\u00e9trica de Fisher**:
   - En este caso especial (Bernoulli), las tres trayectorias en $\eta$ son geom\u00e9tricamente equivalentes (longitudes id\u00e9nticas).
   - En general (multiparam\u00e9trico, no-flat), la $\alpha = 0$ es la \u00fanica geod\u00e9sica en el sentido Riemanniano estricto (m\u00ednima longitud entre pares).

Conexi\u00f3n con el libro:
- §4.2 (definici\u00f3n $\\alpha$-conexi\u00f3n): las tres curvas representadas son las tres conexiones sobre la misma familia.
- §4.3 (dualidad plana): esta familia particular es dualmente plana porque las geod\u00e9sicas son rectas en $(\\eta, \\mu)$.
- §4.4 (Christoffel dual): los s\u00edmbolos de Christoffel $\\Gamma^{(+\\alpha)}$ y $\\Gamma^{(-\\alpha)}$ son los que producen las trayectorias se\u00f1aladas.

Discusi\u00f3n para el LLM mentor:
- ¿C\u00f3mo modificar\u00edas el notebook para la familia Normal($\\mu, \\sigma^2$)? Las geod\u00e9sicas $\\alpha$ son trayectorias distintas en el plano $(\\mu, \\sigma)$ que NO son equivalentes en longitud.
- ¿Por qu\u00e9 la $\\alpha = 0$ pasa por un punto distinto a 0.5 aqu\u00ed? Porque en el plano $(\\eta, p)$, las coordenadas se parametrizan distinto: la recci\u00f3n en $p$ da $p=0.5$, la recci\u00f3n en $\\eta$ da $p=0.5$ (debido a simetr\u00eda), pero el semic\u00edrculo en $\\eta$ da un valor intermedio que depende del arccos promedio.
- ¿Qu\u00e9 significar\u00eda $\\alpha = \\pm 2$ (conexiones extendidas)? La f\u00f3rmula general es interpolaci\u00f3n lineal entre $m$ ($\\alpha=-1$) y $e$ ($\\alpha=+1$); conexiones fuera de $[−1, +1]$ son menos habituales pero formalmente v\u00e1lidas en el marco dual.